# Lab 03 · Grounding on Product Manuals (File Search)

In Lab 01 the persona-only agent was asked about fault code **TU-14** and
**made up the wrong answer** (it guessed "memory/configuration fault" — the manual
actually says *ground-fault protection tripped*). That's the danger of an
ungrounded agent in the field.

Here we fix it with **File Search**: we upload the product manuals into a
**vector store**, attach it to the agent with `FileSearchTool`, and the model
retrieves the real text and **cites** it.

You'll learn to:
1. Create a vector store and upload documents.
2. Attach `FileSearchTool` to an agent.
3. Get grounded, cited answers — and see the hallucination disappear.

> ⚠️ Synthetic training data — not affiliated with or endorsed by Schneider Electric.

## 1. Connect

In [1]:
import sys
from pathlib import Path

here = Path.cwd()
src = next((p / "src" for p in [here, *here.parents] if (p / "src" / "config.py").exists()), None)
if src and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import config

project_client = config.get_project_client()
openai_client = project_client.get_openai_client()
print("Connected. Model:", config.MODEL)

Connected. Model: gpt-5.4


## 2. Create a vector store and upload the manuals

`upload_and_poll` uploads each file and waits for it to be chunked and embedded.
There are 4 synthetic manuals in `data/manuals/`.

> 💡 **If you hit a `403 Forbidden` here:** your account needs the
> **Storage Blob Data Contributor** role on the Foundry project's storage
> account (wait ~5-10 min after granting). This is the most common file-search
> setup issue.

In [2]:
manual_paths = sorted(config.MANUALS_DIR.glob("*.md"))
print("Manuals to upload:", [p.name for p in manual_paths])

vector_store = openai_client.vector_stores.create(name="schneider-manuals")
print("📦 Vector store:", vector_store.id)

for p in manual_paths:
    f = openai_client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id, file=open(p, "rb")
    )
    print(f"  ✅ {p.name} -> {f.status}")

Manuals to upload: ['altivar-atv630.md', 'galaxy-vs-ups.md', 'masterpact-mtz.md', 'powerlogic-pm8000.md']


📦 Vector store: vs_tibmVmDFFxsyQXUTx2Of9XRg


  ✅ altivar-atv630.md -> completed


  ✅ galaxy-vs-ups.md -> completed


  ✅ masterpact-mtz.md -> completed


  ✅ powerlogic-pm8000.md -> completed


## 3. Create a grounded agent

We attach `FileSearchTool(vector_store_ids=[...])` and instruct the agent to
**always search the manuals and cite them**.

In [3]:
from azure.ai.projects.models import FileSearchTool

GROUNDED_INSTRUCTIONS = config.TECH_PERSONA + """

You have a File Search tool over the official product manuals. For any question
about fault codes, specifications, safety, or procedures, ALWAYS search the
manuals first and base your answer on what you find, citing the source. If the
manuals do not contain the answer, say so clearly.
"""

grounded_agent = config.create_prompt_agent(
    project_client,
    name="technician-copilot-grounded",
    instructions=GROUNDED_INSTRUCTIONS,
    tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
)
print(f"✅ Grounded agent: {grounded_agent.name} (v{grounded_agent.version})")

✅ Grounded agent: technician-copilot-grounded (v1)


## 4. Re-ask the question that fooled Lab 01

Compare this answer to the hallucinated one from Lab 01. It should now correctly
say **TU-14 = ground-fault protection tripped**, with a manual citation.

In [4]:
print("🤖", config.ask(project_client, grounded_agent,
    "On a MasterPact MTZ, what does fault code TU-14 mean and what is the first thing I should do?"))

🤖 TU-14 on a MasterPact MTZ means the breaker tripped on ground-fault protection. The first thing to do is investigate the downstream feeder for an insulation fault before reclosing, and log the trip current from the event log. 

Safety first:
- Apply LOTO before servicing.
- Verify arc-flash boundary and required PPE.
- Rack the breaker to the disconnected position before hands-on work where applicable. 

If you want, I can also give you a short field troubleshooting sequence for a TU-14 trip.


## 5. More grounded lookups

In [5]:
for q in [
    "A Galaxy VS UPS is showing fault code E07 — what should I do first?",
    "The PM8000 is reporting A140. Is that a meter failure?",
    "How long must I wait before touching the DC bus terminals on an ATV630?",
]:
    print("👤", q)
    print("🤖", config.ask(project_client, grounded_agent, q), "\n" + "-"*70)

👤 A Galaxy VS UPS is showing fault code E07 — what should I do first?


🤖 First: treat E07 as a serious DC bus fault. On the Galaxy VS, **E07 = DC bus overvoltage detected on the capacitor bank**, and the manual’s **first technician action** is to **transfer the load to static bypass, isolate the UPS, then inspect the DC bus and rectifier IGBT stage before restart** 

Safety before touching anything:
- Apply **LOTO**
- Wear the required **arc-flash PPE** if the cabinet is energized
- Remember the **DC capacitor bank can retain lethal charge for up to 5 minutes after isolation**; **verify zero energy with a meter before contact** 

Practical first steps:
1. **Confirm the load is supported** and transfer to **static bypass** if bypass is available/stable.
2. **Isolate the UPS** from sources per site procedure.
3. **Wait for capacitor discharge** and meter the DC bus to confirm zero energy.
4. Then inspect:
   - **DC bus / capacitor bank**
   - **Rectifier IGBT stage**
5. Do **not** repeatedly restart/reset the unit until the cause is identified 

One note: t

🤖 No — per the PM8000 manual, **A140 is not a meter failure**. It means **Voltage THD exceeds the 8% alarm threshold**. The first recommended action is to **investigate nonlinear loads/VFDs on the feeder** and **confirm the alarm setpoint** against the site power quality standard. 

Practical next checks:
- Look for recent addition/loading of VFDs, UPSs, rectifiers, or other harmonic-producing loads.
- Review the **voltage THD trend** to see if it is sustained or intermittent.
- Verify CT/PT ratios and meter configuration if the reading looks implausible. The manual notes sustained voltage THD above 5% can indicate harmonic-rich loads needing mitigation. 

Safety note: if you’re opening the panel or verifying PT/CT circuits, follow **LOTO** and required **arc-flash PPE**. Also, never open-circuit an energized **CT secondary**; use shorting blocks. 

If you want, I can help you narrow down whether the THD is coming from an upstream source or a local VFD/nonlinear load. 
----------------

🤖 Wait a minimum of 15 minutes before touching the DC bus terminals on an ATV630, and verify zero DC bus voltage with a meter first. The manual also calls out that the DC bus retains lethal voltage after power-off, so apply LOTO and use arc-flash PPE as required  
----------------------------------------------------------------------


## 🙌 Your turn

1. Ask about a spec that is **not** in any manual (e.g. an exact busbar torque
   value). Confirm the grounded agent says it isn't in the manuals rather than
   inventing a number.
2. Ask a question that spans **two** products (e.g. "Which products in the manuals
   mention IGBT stages?") and see how File Search pulls from multiple documents.

In [6]:
# 👉 Your experiment here.

## Clean up

Delete the agent and the vector store so we don't leave resources behind.

In [7]:
config.delete_agent(project_client, grounded_agent)
openai_client.vector_stores.delete(vector_store.id)
print("🗑️  Deleted grounded agent and vector store.")
print("Next: Lab 04 grounds on enterprise search with Azure AI Search.")

🗑️  Deleted grounded agent and vector store.
Next: Lab 04 grounds on enterprise search with Azure AI Search.
